# State Objective Budget-Usage BC Official Replay v1

Goal: turn the existing `state_objective_budget_usage_bc_warm_start_v1` actor artifact into an official qlib replay screen before opening another PPO continuation. This is a model-selection diagnostic, not promotion.

Governance:

- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` and the retention gate from `model_retention_walkforward_gate_v1`.
- Direct comparison baseline: `state_objective_budget_usage_bc_warm_start_v1`, `state_objective_state_conditioned_loss_v1`, and `state_conditioned_checkpoint_replay_v1`.
- Role: `diagnose`; changed layer: `model_selection`, `bc_warm_start`, `official_replay`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`, plus current YTD through `safe_backtest_end`.
- Acceptance: all required WF max drawdowns `< 20%`, YTD return >= SOXLBacktest reference baseline, and YTD return >= 95% of the current best dashboard replay.

Historical contrast:

- Closest success: `state_objective_fold_native_teacher_router_v1` made fold-native targets ready for budget-usage BC training.
- Closest failure: `state_objective_budget_usage_bc_warm_start_v1` fit global BC labels but failed rebound slice readiness, and `state_objective_state_conditioned_loss_v1` improved in-sample rebound fit while failing holdout.
- Why this is not duplicate: this card does not fit another BC grid and does not run a new PPO branch; it official-replays the saved BC actor against the same retention-style gates used for final model retention.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
import sys
sys.path.insert(0, str(ROOT))
CARD = ROOT / "runtime/model_selection/card/state_objective_budget_usage_bc_official_replay_v1"
CARD.mkdir(parents=True, exist_ok=True)

from src.soxl_quant.app.dashboard import write_review_daily_csv
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.data import build_dataset, init_qlib, segment_frame
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.settings import QLIB_DATA_PATH, load_profile

PROFILE = "risk_budget_usage_v1"
SOURCE_CARD = ROOT / "runtime/model_selection/card/state_objective_budget_usage_bc_warm_start_v1"
SOURCE_SELECTION = json.loads((SOURCE_CARD / "selection.json").read_text())
ACTOR_PATH = ROOT / SOURCE_SELECTION["actor_path"]
RETENTION_SUMMARY = json.loads((ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json").read_text())
RETENTION_CONFIG = RETENTION_SUMMARY["summary"]
FRESHNESS = json.loads((ROOT / "dashboard_data/model_replay_freshness.json").read_text())
FRESHNESS_SAFE_BACKTEST_END = FRESHNESS["safe_backtest_end"]
FRESHNESS_LATEST_DATA_DATE = FRESHNESS.get("latest_data_date")
SAFE_BACKTEST_END = FRESHNESS_SAFE_BACKTEST_END
LATEST_DATA_DATE = FRESHNESS_LATEST_DATA_DATE

REFERENCE_RETURN = float(RETENTION_CONFIG["reference_return_pct"])
CURRENT_BEST_RETURN = float(RETENTION_CONFIG["current_best_return_pct"])
REQUIRED_BEST_RETURN = float(RETENTION_CONFIG["required_best_return_pct"])
MAX_WF_DD = float(RETENTION_CONFIG["max_walkforward_drawdown_pct"])
REQUIRED_WF_WINDOWS = list(RETENTION_CONFIG["required_wf_windows"])

context = {
    "experiment": CARD.name,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "profile": PROFILE,
    "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    "actor_path": str(ACTOR_PATH.relative_to(ROOT)),
    "freshness_safe_backtest_end": FRESHNESS_SAFE_BACKTEST_END,
    "freshness_latest_data_date": FRESHNESS_LATEST_DATA_DATE,
    "reference_soxlbacktest_return_pct": REFERENCE_RETURN,
    "current_best_return_pct": CURRENT_BEST_RETURN,
    "required_best_return_pct": REQUIRED_BEST_RETURN,
    "max_walkforward_drawdown_pct": MAX_WF_DD,
    "required_wf_windows": REQUIRED_WF_WINDOWS,
}
(CARD / "context.json").write_text(json.dumps(context, indent=2), encoding="utf-8")
context

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'state_objective_budget_usage_bc_official_replay_v1',
 'created_at': '2026-06-13T13:25:04.916317+00:00',
 'profile': 'risk_budget_usage_v1',
 'source_card': 'runtime/model_selection/card/state_objective_budget_usage_bc_warm_start_v1',
 'actor_path': 'runtime/model/card/risk_budget_usage_v1/soxl_risk_budget_usage_v1_state_objective_budget_usage_bc_warm_start_v1_actor.pt',
 'freshness_safe_backtest_end': '2026-06-11',
 'freshness_latest_data_date': '2026-06-12',
 'reference_soxlbacktest_return_pct': 132.077978,
 'current_best_return_pct': 224.361539,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

## Load Actor And Data

The saved actor is loaded exactly from the BC warm-start card. The profile comes from the repo-owned legacy runtime profile resolver, so the replay uses the same budget-usage mapping, mode router, position cap, and execution settings as `risk_budget_usage_v1`.

In [2]:
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()
test_df = segment_frame(dataset, "test")
price_full = dataset.handler.price_frame.droplevel("instrument")["price"].loc[test_df.index]

profile = load_profile(PROFILE)
ckpt = torch.load(ACTOR_PATH, map_location="cpu", weights_only=False)
_, actor = build_policy(
    int(ckpt["obs_dim"]),
    profile,
    use_moe=bool(ckpt.get("use_moe", False)),
    hard_regime_routing=bool(ckpt.get("use_moe", False)),
)
actor.load_state_dict(ckpt["state_dict"])
actor.eval()

unique_dates = pd.Index(pd.to_datetime(test_df.index).normalize()).unique().sort_values()
ACTUAL_LATEST_DATA_DATE = str(pd.Timestamp(unique_dates[-1]).date())
ACTUAL_SAFE_BACKTEST_END = str(pd.Timestamp(unique_dates[-2]).date())
SAFE_BACKTEST_END = str(min(pd.Timestamp(FRESHNESS_SAFE_BACKTEST_END), pd.Timestamp(ACTUAL_SAFE_BACKTEST_END)).date())
DATA_FRESHNESS_MISMATCH = bool(
    pd.Timestamp(FRESHNESS_SAFE_BACKTEST_END) > pd.Timestamp(ACTUAL_SAFE_BACKTEST_END)
    or pd.Timestamp(FRESHNESS_LATEST_DATA_DATE) > pd.Timestamp(ACTUAL_LATEST_DATA_DATE)
)

context.update({
    "actual_latest_data_date": ACTUAL_LATEST_DATA_DATE,
    "actual_safe_backtest_end": ACTUAL_SAFE_BACKTEST_END,
    "safe_backtest_end_used": SAFE_BACKTEST_END,
    "data_freshness_mismatch": DATA_FRESHNESS_MISMATCH,
})
(CARD / "context.json").write_text(json.dumps(context, indent=2), encoding="utf-8")

data_context = {
    "test_frame_start": str(pd.Timestamp(test_df.index.min()).date()),
    "test_frame_end": str(pd.Timestamp(test_df.index.max()).date()),
    "test_rows": int(len(test_df)),
    "price_start": str(pd.Timestamp(price_full.index.min()).date()),
    "price_end": str(pd.Timestamp(price_full.index.max()).date()),
    "actor_obs_dim": int(ckpt["obs_dim"]),
    "freshness_safe_backtest_end": FRESHNESS_SAFE_BACKTEST_END,
    "freshness_latest_data_date": FRESHNESS_LATEST_DATA_DATE,
    "actual_safe_backtest_end": ACTUAL_SAFE_BACKTEST_END,
    "actual_latest_data_date": ACTUAL_LATEST_DATA_DATE,
    "safe_backtest_end_used": SAFE_BACKTEST_END,
    "data_freshness_mismatch": DATA_FRESHNESS_MISMATCH,
}
(CARD / "data_context.json").write_text(json.dumps(data_context, indent=2), encoding="utf-8")
data_context

[70239:MainThread](2026-06-13 21:25:04,925) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[70239:MainThread](2026-06-13 21:25:05,383) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[70239:MainThread](2026-06-13 21:25:05,383) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/wf2020-bull-diagnostic/data/qlib/soxl')}


[70239:MainThread](2026-06-13 21:25:06,918) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[70239:MainThread](2026-06-13 21:25:06,918) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[70239:MainThread](2026-06-13 21:25:06,919) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


{'test_frame_start': '2020-01-02',
 'test_frame_end': '2026-06-10',
 'test_rows': 1618,
 'price_start': '2020-01-02',
 'price_end': '2026-06-10',
 'actor_obs_dim': 315,
 'freshness_safe_backtest_end': '2026-06-11',
 'freshness_latest_data_date': '2026-06-12',
 'actual_safe_backtest_end': '2026-06-09',
 'actual_latest_data_date': '2026-06-10',
 'safe_backtest_end_used': '2026-06-09',
 'data_freshness_mismatch': True}

In [3]:
WINDOWS = {
    "wf_2020": ("2020-01-01", "2020-12-31"),
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "ytd_2026": ("2026-01-02", SAFE_BACKTEST_END),
}

def _frame_with_following_boundary(start: str, end: str) -> pd.DataFrame:
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    dates = pd.Index(pd.to_datetime(test_df.index).normalize()).unique().sort_values()
    after = dates[dates > end_ts]
    boundary_end = pd.Timestamp(after[0]) if len(after) else pd.Timestamp(dates[-1])
    return test_df.loc[str(start_ts.date()):str(boundary_end.date())].copy()

def _run_window(label: str, start: str, end: str) -> dict:
    frame = _frame_with_following_boundary(start, end)
    price = price_full.loc[frame.index]
    result = backtest_profile(
        actor=actor,
        feature_df=frame,
        price_series=price,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        mode_router_config=profile.mode_router,
        symbol="SOXL",
        asset_idx=0,
        test_seg=(start, end),
        model_config=profile.model,
    )
    daily_path = CARD / f"{label}_daily.csv"
    write_review_daily_csv(
        daily_path,
        result["portfolio"],
        price,
        result.get("position_history", []),
        result.get("trade_signals", []),
    )
    return {
        "window": label,
        "requested_start": start,
        "requested_end": end,
        "effective_start": result["effective_test_seg"][0],
        "effective_end": result["effective_test_seg"][1],
        "return_pct": float(result["return"]),
        "buyhold_return_pct": float(result.get("bnh_return", np.nan)),
        "relative_return_pct": float(result.get("relative_outperformance", np.nan)),
        "sharpe": float(result["sharpe"]),
        "max_dd_pct": float(result["max_dd"]),
        "trades": int(result.get("trades", 0)),
        "avg_position": float(np.mean(result.get("position_history", []) or [0.0])),
        "max_position": float(np.max(result.get("position_history", []) or [0.0])),
        "daily_path": str(daily_path.relative_to(ROOT)),
    }

metrics = [_run_window(label, *seg) for label, seg in WINDOWS.items()]
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(CARD / "official_replay_metrics.csv", index=False)
metrics_df

[70239:MainThread](2026-06-13 21:25:08,346) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[70239:MainThread](2026-06-13 21:25:08,346) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[70239:MainThread](2026-06-13 21:25:08,349) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x131c80d50>


[70239:MainThread](2026-06-13 21:25:08,351) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


[70239:MainThread](2026-06-13 21:25:10,634) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


...

[70239:MainThread](2026-06-13 21:25:17,937) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


[70239:MainThread](2026-06-13 21:25:17,938) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/109 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,window,requested_start,requested_end,effective_start,effective_end,return_pct,buyhold_return_pct,relative_return_pct,sharpe,max_dd_pct,trades,avg_position,max_position,daily_path
0,wf_2020,2020-01-01,2020-12-31,2020-01-02,2020-12-31,16.238860,355.736587,-339.497727,0.716568,21.349092,30,0.240259,0.700000,runtime/model_selection/card/state_objective_b...
1,wf_2022,2022-01-01,2022-12-31,2022-01-03,2022-12-31,-12.139386,-76.614267,64.474882,-0.941319,14.800121,24,0.083689,0.305952,runtime/model_selection/card/state_objective_b...
2,wf_2023_2024,2023-01-01,2024-12-31,2023-01-03,2024-12-31,24.278118,55.654592,-31.376474,0.742131,15.514461,64,0.165252,0.609511,runtime/model_selection/card/state_objective_b...
3,ytd_2026,2026-01-02,2026-06-09,2026-01-02,2026-06-09,27.186052,341.327496,-314.141443,1.718783,17.820794,7,0.244897,0.510147,runtime/model_selection/card/state_objective_b...


In [4]:
gate_rows = []
metrics_by_window = metrics_df.set_index("window").to_dict("index")
for window in REQUIRED_WF_WINDOWS:
    row = metrics_by_window.get(window)
    if row is None:
        gate_rows.append({
            "window": window,
            "gate_type": "walkforward",
            "gate_pass": False,
            "failure": "missing_window",
        })
        continue
    dd = float(row["max_dd_pct"])
    gate_rows.append({
        "window": window,
        "gate_type": "walkforward",
        "return_pct": float(row["return_pct"]),
        "relative_return_pct": float(row["relative_return_pct"]),
        "max_dd_pct": dd,
        "threshold": MAX_WF_DD,
        "gate_pass": bool(dd < MAX_WF_DD),
        "failure": "" if dd < MAX_WF_DD else f"wf_max_drawdown_ge_{MAX_WF_DD:g}",
    })

ytd = metrics_by_window["ytd_2026"]
ytd_return = float(ytd["return_pct"])
ytd_dd = float(ytd["max_dd_pct"])
gate_rows.extend([
    {
        "window": "ytd_2026",
        "gate_type": "soxlbacktest_return_alignment",
        "return_pct": ytd_return,
        "threshold": REFERENCE_RETURN,
        "gate_pass": bool(ytd_return >= REFERENCE_RETURN),
        "failure": "" if ytd_return >= REFERENCE_RETURN else "ytd_return_below_soxlbacktest_baseline",
    },
    {
        "window": "ytd_2026",
        "gate_type": "current_best_return_alignment",
        "return_pct": ytd_return,
        "threshold": REQUIRED_BEST_RETURN,
        "gate_pass": bool(ytd_return >= REQUIRED_BEST_RETURN),
        "failure": "" if ytd_return >= REQUIRED_BEST_RETURN else "ytd_return_below_best_alignment",
    },
])

gate_df = pd.DataFrame(gate_rows)
gate_df.to_csv(CARD / "retention_style_gate.csv", index=False)
failures = [str(v) for v in gate_df.loc[~gate_df["gate_pass"].fillna(False).astype(bool), "failure"].tolist() if str(v)]
all_gate = bool(not failures and set(REQUIRED_WF_WINDOWS).issubset(metrics_by_window.keys()))
selection = {
    "experiment": CARD.name,
    "created_at": context["created_at"],
    "decision": "budget_usage_bc_official_replay_candidate_ready_for_ppo" if all_gate else "budget_usage_bc_official_replay_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": bool(all_gate),
    "profile": PROFILE,
    "actor_path": str(ACTOR_PATH.relative_to(ROOT)),
    "source_experiment": "state_objective_budget_usage_bc_warm_start_v1",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [
        "state_objective_budget_usage_bc_warm_start_v1",
        "state_objective_state_conditioned_loss_v1",
        "state_conditioned_checkpoint_replay_v1",
        "model_retention_walkforward_gate_v1",
    ],
    "reference_baseline": "soxlbacktest_qlib_buysell_reference",
    "freshness_safe_backtest_end": FRESHNESS_SAFE_BACKTEST_END,
    "actual_safe_backtest_end": ACTUAL_SAFE_BACKTEST_END,
    "safe_backtest_end": SAFE_BACKTEST_END,
    "data_freshness_mismatch": DATA_FRESHNESS_MISMATCH,
    "required_soxlbacktest_return_pct": REFERENCE_RETURN,
    "required_best_return_pct": REQUIRED_BEST_RETURN,
    "target_max_dd_pct": MAX_WF_DD,
    "required_windows": REQUIRED_WF_WINDOWS,
    "failures": failures,
    "all_gate_pass": all_gate,
    "wf_pass_count": int(gate_df.loc[gate_df["gate_type"].eq("walkforward"), "gate_pass"].fillna(False).astype(bool).sum()),
    "ytd_return_pct": ytd_return,
    "ytd_max_dd_pct": ytd_dd,
    "wf_max_dd_pct": {w: float(metrics_by_window[w]["max_dd_pct"]) for w in REQUIRED_WF_WINDOWS if w in metrics_by_window},
    "wf_return_pct": {w: float(metrics_by_window[w]["return_pct"]) for w in REQUIRED_WF_WINDOWS if w in metrics_by_window},
    "promotion_blocker": "Official replay is a diagnostic of a BC warm-start actor. Full retention still requires a trained retained actor bundle plus fixed-split, all WF, YTD alignment, and dashboard replay evidence.",
    "next_mechanism": "run_fold_local_ppo_only_if_ytd_and_wf_gap_is_not_structural_else_revise_targets",
    "sidecars": [
        "context.json",
        "data_context.json",
        "official_replay_metrics.csv",
        "retention_style_gate.csv",
        "selection.json",
        "summary.json",
        "conclusion.md",
        *[f"{label}_daily.csv" for label in WINDOWS],
    ],
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
gate_df

,window,gate_type,return_pct,relative_return_pct,max_dd_pct,threshold,gate_pass,failure
0,wf_2020,walkforward,16.238860,-339.497727,21.349092,20.000000,False,wf_max_drawdown_ge_20
1,wf_2022,walkforward,-12.139386,64.474882,14.800121,20.000000,True,
2,wf_2023_2024,walkforward,24.278118,-31.376474,15.514461,20.000000,True,
3,ytd_2026,soxlbacktest_return_alignment,27.186052,NaN,NaN,132.077978,False,ytd_return_below_soxlbacktest_baseline
4,ytd_2026,current_best_return_alignment,27.186052,NaN,NaN,213.143462,False,ytd_return_below_best_alignment


In [5]:
summary = {
    "context": context,
    "data_context": data_context,
    "metrics": metrics_df.to_dict("records"),
    "gate": gate_df.to_dict("records"),
    "selection": selection,
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

wf_summary = "; ".join(
    f"{w} return {metrics_by_window[w]['return_pct']:.2f}%, MaxDD {metrics_by_window[w]['max_dd_pct']:.2f}%"
    for w in REQUIRED_WF_WINDOWS
    if w in metrics_by_window
)
conclusion_lines = [
    "# Conclusion",
    "",
    f"Decision: `{selection['decision']}`.",
    "",
    f"Official replayed `{PROFILE}` BC warm-start actor from `{ACTOR_PATH.relative_to(ROOT)}` through actual safe end `{SAFE_BACKTEST_END}`.",
    f"Freshness file safe end `{FRESHNESS_SAFE_BACKTEST_END}` / latest `{FRESHNESS_LATEST_DATA_DATE}`; current qlib data safe end `{ACTUAL_SAFE_BACKTEST_END}` / latest `{ACTUAL_LATEST_DATA_DATE}`; mismatch `{DATA_FRESHNESS_MISMATCH}`.",
    f"Walk-forward readout: {wf_summary}.",
    f"YTD return `{ytd_return:.2f}%`, MaxDD `{ytd_dd:.2f}%`; required SOXLBacktest baseline `{REFERENCE_RETURN:.2f}%`, current-best alignment `{REQUIRED_BEST_RETURN:.2f}%`.",
    f"Failures: `{';'.join(failures) if failures else 'none'}`.",
    "",
    "This card does not promote or retain an actor. It decides whether the existing budget-usage BC actor is strong enough to justify a fold-local PPO continuation under the current retention-style gates.",
]
(CARD / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")
print("\n".join(conclusion_lines))

# Conclusion

Decision: `budget_usage_bc_official_replay_no_promotion`.

Official replayed `risk_budget_usage_v1` BC warm-start actor from `runtime/model/card/risk_budget_usage_v1/soxl_risk_budget_usage_v1_state_objective_budget_usage_bc_warm_start_v1_actor.pt` through actual safe end `2026-06-09`.
Freshness file safe end `2026-06-11` / latest `2026-06-12`; current qlib data safe end `2026-06-09` / latest `2026-06-10`; mismatch `True`.
Walk-forward readout: wf_2020 return 16.24%, MaxDD 21.35%; wf_2022 return -12.14%, MaxDD 14.80%; wf_2023_2024 return 24.28%, MaxDD 15.51%.
YTD return `27.19%`, MaxDD `17.82%`; required SOXLBacktest baseline `132.08%`, current-best alignment `213.14%`.
Failures: `wf_max_drawdown_ge_20;ytd_return_below_soxlbacktest_baseline;ytd_return_below_best_alignment`.

This card does not promote or retain an actor. It decides whether the existing budget-usage BC actor is strong enough to justify a fold-local PPO continuation under the current retention-style gates.


## Single-Run Conclusion

The executable conclusion is written to `conclusion.md` and `selection.json`. Use this card as a gate before spending another PPO run on the budget-usage BC warm-start branch.